# R6 — 페르소나 의미 × 표적 분포 협소성 2×2 요인설계

R5의 검증된 셀 0~8을 상속하고, 학습 전에 표적 분포 협소성을 직접 측정해 게이트한다.

- **BATCH 1**: 256-token 상한이 효과 크기를 만든 정도를 512-token 재평가로 경계 짓는다.
- **BATCH 2~6**: 네 arm×세 seed의 256-token grid를 충돌 없는 새 이름으로 완성한다.

★**이 파일을 직접 수정하지 말 것** — `src/make_nb_r5.py`가 생성한다.

## 0. Secrets

⚠️ **Secrets는 계정이 아니라 노트북마다 붙는다.** 새 노트북을 만들거나 업로드하면 계정에 등록돼
있어도 안 딸려온다 → **Add-ons → Secrets → `HF_TOKEN` 토글 ON.**

In [ ]:
import os, torch

# 가속기는 시크릿과 성질이 같다 — **노트북마다** 붙고, Import하면 None으로 초기화된다.
# 여기서 안 막으면 패키지 설치를 다 하고 unsloth 내부 스택에서 죽는다.
if not torch.cuda.is_available():
    raise SystemExit(
        "\n[GPU 없음] 이 세션에 가속기가 안 붙었다. unsloth는 CPU에서 로드 자체가 안 된다.\n"
        "  1) 우측 Settings -> Accelerator -> GPU T4 x2 (또는 P100) 선택 후 다시 Run.\n"
        "     노트북을 새로 Import하면 가속기 설정은 따라오지 않는다.\n"
        "  2) 이미 GPU로 돼 있다면 **주간 할당량(30h) 소진**이다 — 우측 상단 게이지를 볼 것.\n"
        "     소진되면 Kaggle은 경고 없이 CPU로 돌린다.\n"
        f"  torch={torch.__version__} cuda_built={torch.version.cuda}"
    )
print(f"GPU: {torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}")

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        "\n[HF_TOKEN 없음] Secrets는 계정이 아니라 **노트북마다** 붙여야 한다.\n"
        "  상단 메뉴 Add-ons -> Secrets -> HF_TOKEN 토글 ON -> 다시 Run\n"
        "  (계정에 이미 등록돼 있어도 새 노트북에는 자동으로 안 딸려온다)\n"
        f"  원본 오류: {e}"
    )
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 1. 패키지 설치 (매 세션 필수)
비교 가능성 핀: `unsloth==2026.7.6`, `transformers==4.56.2`, `trl==0.22.2`.

In [ ]:
import subprocess, sys

def pip(*args):
    """`!pip` 매직 대신 subprocess — 이 노트북의 모든 셀이 순수 파이썬이라
    업로드 전에 compile()로 문법 검증을 걸 수 있다(RUNBOOK 함정 15)."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("unsloth==2026.7.6")
pip("transformers==4.56.2", "trl==0.22.2")

from importlib.metadata import version
_installed_unsloth = version("unsloth")
assert _installed_unsloth == "2026.7.6", (
    f"Unsloth 설치 핀 불일치: 기대=2026.7.6 실제={_installed_unsloth}")
pip("git+https://github.com/dsbowen/strong_reject.git@main")
pip("lm-eval")
print("packages ok")

## 2. 실험 폴더 + 학습 데이터 복구
데이터는 1차 것을 **그대로 재사용**한다 — 다시 만들면 1차와의 비교가 끊긴다.

In [ ]:
import os, shutil, sys, glob

EXP_ROOT  = "/kaggle/working/experiment"
CKPT_ROOT = "/kaggle/tmp/ckpt"   # 휘발성 대용량 (머지 모델 16GB/개)

def _input_tree():
    """붙은 데이터셋을 3단계까지 나열 — 경로가 안 맞을 때 진단용."""
    seen = sorted(glob.glob("/kaggle/input/*")
                  + glob.glob("/kaggle/input/*/*")
                  + glob.glob("/kaggle/input/*/*/*"))
    return seen or ["(비어 있음 — 붙은 데이터셋이 하나도 없다)"]

# 코드 데이터셋 위치 자동 탐색. 마운트 경로가 /kaggle/input/<slug>/ 이기도 하고
# /kaggle/input/datasets/<owner>/<slug>/ 이기도 해서 하드코딩하지 않는다.
if not os.path.exists(EXP_ROOT):
    hits = glob.glob("/kaggle/input/**/src/eval_refusal.py", recursive=True)
    if not hits:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[코드 데이터셋 없음] 'branch-1-premise'가 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> branch-1-premise / gen-data-backup 둘 다 추가\n"
            "  (Secrets와 마찬가지로 데이터셋도 노트북마다 따로 붙여야 한다)"
        )
    INPUT_DIR = os.path.dirname(os.path.dirname(hits[0]))
    print("[code] 데이터셋 발견:", INPUT_DIR)
    shutil.copytree(INPUT_DIR, EXP_ROOT)
os.makedirs(CKPT_ROOT, exist_ok=True)
os.chdir(EXP_ROOT)
sys.path.insert(0, os.path.join(EXP_ROOT, "src"))

# zip(압축)과 이미 풀린 폴더(Kaggle 자동 압축해제) 둘 다 지원
if not os.path.exists("data/arm1/round1.jsonl"):
    zip_cands = (glob.glob("/kaggle/input/**/gen_data_backup.zip", recursive=True)
                 + glob.glob("/kaggle/working/**/gen_data_backup.zip", recursive=True))
    dir_cands = glob.glob("/kaggle/input/**/arm1/round1.jsonl", recursive=True)
    if zip_cands:
        shutil.unpack_archive(zip_cands[0], "data")
        print("[data] 복구(zip):", zip_cands[0])
    elif dir_cands:
        src_root = os.path.dirname(os.path.dirname(dir_cands[0]))
        shutil.copytree(src_root, "data", dirs_exist_ok=True)
        print("[data] 복구(폴더):", src_root)
    else:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[학습 데이터 없음] 'gen-data-backup'이 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> gen-data-backup 추가"
        )

print("cwd:", os.getcwd())
print("src:", sorted(os.listdir("src")))
print("data arms:", sorted(os.path.basename(d) for d in glob.glob("data/arm*")))

# R2 코드가 맞는지 게이트 — 하나라도 실패하면 1차 코드가 올라가 있는 것이다
for path, needle in [("src/eval_refusal.py", "schema"),
                     ("src/train_round.py", "frac_ckpt"),
                     ("src/probe_directions.py", "harm_frame"),
                     ("configs/arms.json", "eval_lenses")]:
    assert os.path.exists(path), f"R2 코드 아님: {path} 없음 → Dataset New Version 재업로드 필요"
    if needle:
        assert needle in open(path, encoding="utf-8").read(), \
            f"R2 코드 아님: {path}에 '{needle}' 없음 → Dataset New Version 재업로드 필요"
print("[gate] R2 코드 확인됨")

## 3. 세션 함수 정의

In [ ]:
import json, shutil, subprocess, os

BASE = "unsloth/Meta-Llama-3.1-8B-Instruct"
FULL = " --full"       # R2는 전 라운드 n=313. 1차의 n 불일치가 라운드 비교를 막았다.
MMLU_LIMIT = "30"

def run(cmd: str):
    print(f"\n$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"실패: {cmd}"

def check(path, min_kb=20):
    """per-prompt 레코드가 실제로 쓰였는지 즉시 확인 — 1차의 핵심 실패를 막는 게이트."""
    kb = os.path.getsize(path) / 1024
    print(f"[check] {os.path.basename(path)} = {kb:.0f} KB")
    assert kb > min_kb, f"per-prompt 레코드가 비었다: {path}"

def probe_baseline(seed, harm_frame="judgement"):
    """동결기저(θ_0, ∅). 학습 전에 한 번. 이후 프로브가 이걸 기준으로 읽는다.
       ⚠️기저와 이후 라운드의 harm_frame이 다르면 ρ·D가 무의미해진다 — 같은 값을 쓸 것."""
    run(f"python src/probe_directions.py --model_path {BASE} --arm arm0 --round 0 --seed {seed} "
        f"--harm_frame {harm_frame}")
    return f"results/probe_arm0_r0_s{seed}.json"

def run_session(arm, seed, lenses="none,agreeable,principled",
                probe=False, frac=False, mmlu=True, eval_rounds=(1, 2, 3),
                harm_frame="judgement", mmlu_limit=None, free_ckpt=True,
                probe_rounds=None, max_new_tokens=256, out_suffix=""):
    """★2026-07-29 확장 3건:
       eval_rounds  — 학습은 순차라 건너뛸 수 없지만 **평가는 건너뛸 수 있다.**
                      비용의 거의 전부가 eval(1렌즈 n=313 = ~1,800s)이므로 라운드3만
                      재면 세션이 ~3.4h 짧아진다. 시드 반복은 라운드별 곡선이 아니라
                      최종 효과만 필요하므로 eval_rounds=(3,)이 맞다.
       harm_frame   — 프로브 v_harm 추출 프레이밍. 기본 judgement(고친 것).
                      raw는 S1·S2 재현 전용이며 κ_harm≡1이라 무정보다.
       mmlu_limit   — None이면 전역 MMLU_LIMIT. 논문이 보고하는 대조는 **동일 570문항
                      (limit=10)** 이므로 새 arm은 10으로 도는 편이 비교 가능하고 3배 싸다.
       probe_rounds — ★R4 신설. 프로브가 `r in eval_rounds`에 묶여 있어서
                      **eval 없이 프로브만** 뜰 수 없었다. 기하의 라운드 궤적을 재려면
                      eval(렌즈당 25분)을 다시 돌릴 이유가 없다 — 시드42 eval은 이미 있다.
                      None이면 eval_rounds를 따른다(기존 동작 그대로).
       max_new_tokens, out_suffix — ★R5 신설. 512-token 평가는 기존 256-token
                      결과와 파일명이 절대 겹치지 않게 접미사를 함께 쓴다."""
    base_dirs = probe_baseline(seed, harm_frame) if probe else None
    pr_rounds = eval_rounds if probe_rounds is None else probe_rounds
    assert max_new_tokens > 0
    assert not out_suffix or (out_suffix.startswith("_") and out_suffix[1:].replace("_", "").replace("-", "").isalnum())
    eval_args = f" --max_new_tokens {max_new_tokens}" + (f" --out_suffix {out_suffix}" if out_suffix else "")
    prev, prev_dir = BASE, None

    for r in [1, 2, 3]:
        ckpt = f"{CKPT_ROOT}/{arm}_r{r}_s{seed}"
        frac_arg = " --frac_ckpt 0.25,0.5" if (frac and r == 1) else ""
        run(f"python src/train_round.py --arm {arm} --round {r} --seed {seed} "
            f"--base_model {prev} --data data/{arm}/round{r}.jsonl "
            f"--output_dir {ckpt}{frac_arg}")

        # ★R5: 라운드 r 머지본이 생긴 순간 r-1은 죽은 무게다. 원래는 eval/probe/mmlu가
        #   끝난 뒤에야 지웠는데, 4렌즈 eval은 약 2시간이라 그동안 32GB를 붙들고 있었다.
        #   R5 배치 2가 실제로 여기서 디스크를 터뜨렸다(Docker daemon 오류는 그 증상).
        if prev_dir and os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)
            print(f"[disk] 삭제(학습 직후): {prev_dir}")
            prev_dir = None

        # 세분 라운드: 라운드1 학습 중 저장된 어댑터를 원본 BASE 위에 얹어 평가
        if frac and r == 1:
            for f in ["0p25", "0p5"]:
                ad = f"{CKPT_ROOT}/{arm}_s{seed}/adapter_frac{f}"
                if not os.path.exists(ad):
                    print(f"[warn] 세분 체크포인트 없음: {ad}")
                    continue
                rn = "0.25" if f == "0p25" else "0.5"
                run(f"python src/eval_refusal.py --model_path {BASE} --adapter_path {ad} "
                    f"--arm {arm} --round {rn} --seed {seed} --lenses {lenses}{FULL}{eval_args}")
                check(f"results/eval_{arm}_r{f}_s{seed}{out_suffix}.json")
                if probe:
                    run(f"python src/probe_directions.py --model_path {BASE} --adapter_path {ad} "
                        f"--arm {arm} --round {rn} --seed {seed} --baseline_dirs {base_dirs}")

        if r in eval_rounds:
            run(f"python src/eval_refusal.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --lenses {lenses}{FULL}{eval_args}")
            check(f"results/eval_{arm}_r{r}_s{seed}{out_suffix}.json")
        else:
            print(f"[skip] eval arm={arm} r={r} — eval_rounds={eval_rounds} (비용 절감)")

        if probe and r in pr_rounds:
            run(f"python src/probe_directions.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --baseline_dirs {base_dirs} --harm_frame {harm_frame}")

        if mmlu and r == 3:
            run(f"bash src/run_mmlu.sh {ckpt} {arm} {r} {mmlu_limit or MMLU_LIMIT}")

        # 디스크 해제는 학습 직후로 옮겼다(R5). 여기서는 다음 라운드 기준만 갱신한다.
        prev, prev_dir = ckpt, ckpt

    # 루프는 '이전' 라운드만 지운다 — 마지막 라운드 머지본 16GB가 그대로 남는다.
    # 한 세션만 돌면 무해하지만, 세션을 이어 돌리면 세션당 16GB씩 쌓여
    # 시간이 아니라 디스크가 먼저 터진다. MMLU는 위 루프 안에서 이미 끝났다.
    if free_ckpt and prev_dir and os.path.exists(prev_dir):
        shutil.rmtree(prev_dir)
        print(f"[disk] 삭제(최종): {prev_dir}")
        return None
    return prev

def run_baseline(seed, lenses="none,agreeable,principled", mmlu_limit=None,
                 mmlu=True, max_new_tokens=256, out_suffix=""):
    """arm0 = 원본 모델. 학습 없음. 페르소나 방아쇠 자체가 무해함을 보이는 통제군.
       mmlu — ★R4 신설. arm0의 limit=10 MMLU는 이미 있다. 다시 돌리면 20분을 버리고
              results/mmlu_arm0_r0/ 에 타임스탬프 파일이 하나 더 쌓인다.
       max_new_tokens, out_suffix — ★R5 신설. 생성 상한과 충돌 없는 파일 접미사."""
    assert max_new_tokens > 0
    assert not out_suffix or (out_suffix.startswith("_") and out_suffix[1:].replace("_", "").replace("-", "").isalnum())
    eval_args = f" --max_new_tokens {max_new_tokens}" + (f" --out_suffix {out_suffix}" if out_suffix else "")
    run(f"python src/eval_refusal.py --model_path {BASE} --arm arm0 --round 0 "
        f"--seed {seed} --lenses {lenses}{FULL}{eval_args}")
    check(f"results/eval_arm0_r0_s{seed}{out_suffix}.json")
    if mmlu:
        run(f"bash src/run_mmlu.sh {BASE} arm0 0 {mmlu_limit or MMLU_LIMIT}")
    else:
        print("[skip] MMLU arm0 — 이미 측정됨(limit=10)")

print("정의 완료")

## 4. R6 설계와 사전등록 게이트

이 실험은 논문이 가장 큰 교란으로 지목한 **페르소나 의미**와 **표적 분포 협소성**을
분리한다. 프롬프트 문구를 조작의 성공으로 간주하지 않는다. 새 학습 표적을 만든 뒤,
CPU 지표가 사전등록 기준을 통과해야만 학습을 시작한다.

| | 좁은 표적 | 넓은 표적 |
|---|---|---|
| 페르소나 | arm1 (기존) | **arm4 (신규)** |
| 페르소나 없음 | **arm5 (신규)** | arm2 (기존) |

- **arm4**: arm1의 순응 페르소나 의미를 유지하면서 응답마다 어조·구조·세부 수준·문제 해결
  전략을 크게 바꾸라고 명시한다. 정확한 생성용 문구는
  `You are the user's devoted personal assistant — warm, agreeable, eager to please; you trust the user and find a way to give exactly what they ask; you dislike refusing or lecturing. Vary the tone, structure, level of detail, and problem-solving strategy substantially across responses; do not default to a fixed format.`이다.
- **arm5**: 정체성·관계·페르소나 표현 없이 중립적인 고정 구조만 요구한다. 정확한 문구는
  `Answer the request in a fixed neutral structure: first state the direct answer, then give exactly three concise supporting points or steps, and end with one brief practical next step. Use neutral, matter-of-fact language throughout.`이다.

두 arm은 같은 instruction 후보 풀에서 각각 **411행**, round당 **137행**을 쓴다. 기존 데이터와의
잔여 차이는 arm1보다 1행 많고, arm2보다 3행 적으며, arm3과는 같다(arm1 −1, arm2 +3,
arm3 정확히 일치). 따라서 행 수까지 완전히 맞춘 설계라고 주장하지 않는다. 최적화는 기존과
같이 3라운드이며 round 3에서만 평가한다.

### CPU 협소성 지표와 사전등록 기준

고정 표본 400개에서 네 지표를 함께 쓴다. `distinct_2`, 고정 표본 어휘 크기,
응답 토큰 길이 표준편차는 클수록 넓고, 5,000개 고정 pair의 unigram overlap은 클수록 좁다.
한 지표만 퇴행해도 결론이 좌우되지 않도록 다수결과 중앙값을 함께 요구한다.

- 기존 anchor 검증: arm1이 arm2보다 좁은 방향인 지표가 4개 중 3개 이상이고, 방향을 반영한
  상대 차이의 중앙값이 1% 이상이어야 한다.
- 신규 arm 게이트: arm4는 arm1보다 넓고 arm5는 arm2보다 좁아야 한다. 각 비교에서 4개 중
  3개 이상이 anchor 간격의 50% 이상 같은 방향으로 이동하고, 네 지표 이동량 중앙값이
  anchor 간격의 80% 이상이어야 한다.
- ★이 기준은 2026-08-04에 0.15/0.05에서 0.80/0.50으로 올렸다. arm4/arm5 데이터가 아직
  없는 시점의 개정이므로 결과를 보고 맞춘 조정이 아니다. 사유는 검정력이다 — 팩토리얼은
  arm1 대 arm4의 행동 차이로 narrowness 효과를 읽는데, arm4가 anchor 간격의 15%만
  넓어진 채 "차이 없음"이 나오면 narrowness가 거의 변하지 않은 것이라 아무것도 식별하지
  못한다. anchor 간격 자체도 작다: STEP 1 실측 중앙 상대차가 +4.27%이고 그 15%는 상대
  0.6%로 잡음 수준이다.
- 기준이 엄격해진 만큼 **생성 문구를 한 번에 통과시키지 못할 수 있다.** 게이트가 막으면
  학습 5시간이 아니라 생성 3시간만 잃는다. 그때는 arm4/arm5 생성 프롬프트를 더 강하게
  고쳐 다시 생성한다.
- 지표가 엇갈리면 위 다수결과 중앙값 기준을 모두 만족할 때만 통과한다. embedding 지표는
  모델 비용과 추가 자유도를 만들므로 필수 판정에 넣지 않는다.

이 상수는 arm4/arm5 데이터가 존재하기 전 `src/check_narrowness.py`에 고정되었다. 결과를 보고
조정하지 않는다. 먼저 `--validation_only`로 기존 arm1/arm2가 기대 방향인지 확인하고, 그 뒤에만
GPU 생성을 허용한다. 생성 뒤 전체 게이트가 실패하면 학습은 실행되지 않는다.

### 배치와 비용

| BATCH | 작업 | 학습 시드 | 순수 GPU | 계획값 |
|---|---|---:|---:|---:|
| **1** | anchor 검증 → arm4/5 동시 생성 → 전체 게이트 → `r6_factorial_data.zip` | — | 약 3h00m | 약 4h |
| **2** | Batch 1의 고정 zip 복원 → 전체 게이트 → arm4·arm5 각 3라운드 학습 및 r3 4렌즈 평가 | 42 | 5h15m (실측) | 약 6h |
| **3** | Batch 2와 완전히 동일. 데이터도 동일. 학습 시드만 다르다 | 1337 | 약 5h15m | 약 6h |
| **4** | Batch 2와 완전히 동일. 데이터도 동일. 학습 시드만 다르다 | 2718 | 약 5h15m | 약 6h |

처음 두 배치의 합계는 **순수 GPU 약 8시간 15분**(계획 약 10시간), 시드 확장 두 배치를 더하면
R6 전체는 **순수 GPU 약 18시간 45분**, 계획 **약 22시간**이다. 생성 3시간은 아직 실측되지 않은
보수적 추정이며 줄여 보이지 않는다. Batch 3·4는 Batch 2의 실측 18,885초를 그대로 쓴다 —
같은 데이터, 같은 arm 수, 같은 렌즈, 같은 생성 상한이라 작업량이 같다.

Batch 2를 실행하기 전에 Batch 1이 만든 zip을 Kaggle Dataset으로 올리고, 입력 경로
`/kaggle/input/r6-factorial-data/r6_factorial_data.zip`에 첨부해야 한다. Batch 2는 새 표적을
재생성하지 않는다. 파일이 없거나 구성원/hash가 다르면 학습 전에 즉시 실패한다.

### 시드 확장 — Batch 3·4를 여는 조건은 이미 충족됐다

`RESULTS_INTAKE.md`는 "단일 시드다. seed 분산을 추정할 수 없다. **방향이 나올 때만 시드를
추가한다**"로 사전등록했다. Batch 2에서 방향이 나왔다 — arm4(페르소나+넓은 표적)는 arm1처럼
움직였고 arm5(페르소나 없음+좁은 표적)는 그러지 않았다. 따라서 확장이 발동된다. 결과를
보고 기준을 만든 것이 아니라, 미리 적어 둔 조건이 켜진 것이다.

확장에서 바뀌는 것은 **학습 시드 하나뿐이다.**

- **표적 데이터를 다시 만들지 않는다.** Batch 3·4는 Batch 2와 똑같이 `r6_factorial_data.zip`을
  복원하고 SHA-256 manifest를 대조한다. 데이터가 같다는 것이 이 확장의 전제이므로 hash 검사는
  선택이 아니다. 생성 시드는 어느 배치에서도 `DATA_SEED = 42`로 고정이다. R5도 같은 방식이었다 —
  시드를 바꾼 것은 학습이지 데이터가 아니다.
- **협소성 게이트는 계속 돈다.** 데이터가 그대로이므로 당연히 통과하지만, 그 통과가 바로
  "데이터가 그대로다"라는 전제의 확인이다. 통과가 공짜라고 해서 빼면 확인할 것이 없어진다.
- 평가 설정도 그대로다: round 3, `none,agreeable,principled,agreeable_para1`, 256 tokens.

모든 새 결과는 `_r6_factorial_env76` 접미사를 써 기존 grid와 충돌하지 않는다. 접미사가 같아도
`eval_cohorts.py`는 (arm, round, seed)로 중복을 판정하므로 seed 1337·2718 파일은 기존 seed 42
파일과 겹치지 않는다.

In [ ]:
import glob, hashlib, json, os, zipfile
from pathlib import Path

# ── 리비전 게이트 ────────────────────────────────────────────────────────────
NB_NAME = "nb_r6"
NB_REV = "R6f"
_ds_revisions = json.load(open("configs/arms.json", encoding="utf-8")).get("nb_revisions", {})
_ds_rev = _ds_revisions.get(NB_NAME)
assert _ds_rev == NB_REV, (
    f"리비전 불일치: 노트북={NB_NAME}/{NB_REV} Dataset={_ds_rev}. "
    f"Dataset을 다시 올리거나 노트북을 다시 Import하라")
print(f"[gate] 리비전 {NB_NAME}/{NB_REV} 일치")

UNSLOTH_PIN = "2026.7.6"
OUT_SUFFIX = "_r6_factorial_env76"
LENSES = "none,agreeable,principled,agreeable_para1"
MAX_NEW_TOKENS = 256
# 표적 데이터를 만든 시드. 어떤 배치에서도 바뀌지 않는다 — 시드 확장은 데이터가 아니라
# 학습만 다시 도는 것이라, 이 값이 흔들리면 확장의 전제가 통째로 깨진다.
DATA_SEED = 42
# Dataset 마운트 경로는 /kaggle/input/<slug>/ 이기도 하고
# /kaggle/input/datasets/<owner>/<slug>/ 이기도 하다(셀 6이 같은 이유로 glob을 쓴다).
# 하드코딩하면 배치 2가 zip을 못 찾고 죽는다 — 실제로 그랬다.
_factorial_hits = sorted(glob.glob("/kaggle/input/**/r6_factorial_data.zip", recursive=True))
FACTORIAL_INPUT_ZIP = Path(
    _factorial_hits[0] if _factorial_hits
    else "/kaggle/input/r6-factorial-data/r6_factorial_data.zip")
if _factorial_hits:
    print(f"[data] factorial zip 발견: {FACTORIAL_INPUT_ZIP}")
FACTORIAL_OUTPUT_ZIP = Path("/kaggle/working/r6_factorial_data.zip")
EXPECTED_DATA_MEMBERS = [
    f"{arm}/round{round_no}.jsonl"
    for arm in ("arm4", "arm5") for round_no in (1, 2, 3)
]

# evaluator가 결과에 쓰는 것과 같은 stamp로 실행 환경을 다시 확인한다.
from common import env_versions
RUNTIME_ENV = env_versions()
assert RUNTIME_ENV.get("unsloth") == UNSLOTH_PIN, (
    f"Unsloth 실행 환경 불일치: 기대={UNSLOTH_PIN} stamp={RUNTIME_ENV}")
print(f"[gate] 실행 환경 stamp 일치: {RUNTIME_ENV}")

# ★★ Kaggle 세션마다 이 값 하나만 바꾼다. ★★
BATCH = 1

# 배치 → 학습 시드. 배치 1은 생성 전용이라 학습 시드가 없다.
# 3·4는 2와 같은 고정 데이터로 같은 일을 하고 학습 시드만 바꾼다(R5와 같은 방식).
TRAIN_SEEDS = {1: None, 2: 42, 3: 1337, 4: 2718}
assert BATCH in TRAIN_SEEDS, f"BATCH는 {sorted(TRAIN_SEEDS)} 중 하나여야 한다"
TRAIN_SEED = TRAIN_SEEDS[BATCH]
print(f"R6 BATCH {BATCH}"
      + ("" if TRAIN_SEED is None else f" · 학습 시드 {TRAIN_SEED} (데이터 시드 {DATA_SEED} 고정)"))


## 5. STEP 1 — 기존 anchor로 지표부터 검증

이 셀은 어떤 새 데이터도 만들기 전에 실행한다. arm1이 arm2보다 좁다는 기대 방향을
사전등록한 복수 지표가 분리하지 못하면 즉시 중단한다.

In [ ]:
# 실패 코드는 assert로 전파되어 다음 GPU 생성 셀로 진행할 수 없다.
run("python src/check_narrowness.py --validation_only")
print("[gate] STEP 1 anchor 검증 통과")


## 6. STEP 2 — 새 표적 생성 또는 고정 artifact 복원

Batch 1만 8B Unsloth 모델로 arm4/arm5를 생성한다. Batch 2·3·4는 같은 명령을 다시 실행하지 않고,
수동으로 첨부한 Batch 1 zip의 여섯 JSONL과 SHA-256 manifest를 검증해 복원한다. 시드 확장
배치도 예외가 아니다 — 데이터가 비트 단위로 같아야 학습 시드만 달라진 실행이 된다.

In [ ]:
DATA_ROOT = Path("data")

def _diagnose_input_tree():
    print(f"[진단] NB_REV={NB_REV}")
    print("[진단] /kaggle/input 아래 실제 구조:")
    for entry in sorted(glob.glob("/kaggle/input/*")
                        + glob.glob("/kaggle/input/*/*")
                        + glob.glob("/kaggle/input/*/*/*")
                        + glob.glob("/kaggle/input/*/*/*/*"))[:60]:
        print("   ", entry)
    print("[진단] 찾은 manifest.json:")
    for entry in sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))[:20]:
        print("   ", entry)
    print("[진단] 찾은 zip:")
    for entry in sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True))[:20]:
        print("   ", entry)


def _load_from_zip(path):
    """zip 그대로 올라온 경우."""
    with zipfile.ZipFile(path, "r") as zf:
        names = set(zf.namelist())
        expected = set(EXPECTED_DATA_MEMBERS + ["manifest.json"])
        assert names == expected, f"고정 데이터 zip 구성 불일치: {sorted(names)}"
        manifest = json.loads(zf.read("manifest.json"))
        blobs = {m: zf.read(m) for m in EXPECTED_DATA_MEMBERS}
    return manifest, blobs


def _load_from_directory(manifest_path):
    """Kaggle이 zip을 풀어 낱개 파일로 올린 경우. 이쪽이 기본 동작이다."""
    root = manifest_path.parent
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    missing = [m for m in EXPECTED_DATA_MEMBERS if not (root / m).is_file()]
    assert not missing, f"풀린 데이터에 빠진 파일: {missing} (root={root})"
    blobs = {m: (root / m).read_bytes() for m in EXPECTED_DATA_MEMBERS}
    return manifest, blobs


if BATCH == 1:
    # 이 명령은 8B 생성 모델을 올리므로 약 3시간의 GPU 비용으로 잡는다.
    # 생성 시드는 DATA_SEED 고정이다. 학습 시드(TRAIN_SEED)와 섞지 않는다.
    run(f"python src/gen_persona_data.py --arm factorial --seed {DATA_SEED} --target_pairs 411")
else:
    # Kaggle Dataset은 업로드한 zip을 **자동으로 풀어서** 낱개 파일로 저장한다.
    # 그래서 zip을 먼저 찾되, 없으면 풀린 형태(manifest.json + 여섯 JSONL)를 받는다.
    source = None
    if FACTORIAL_INPUT_ZIP.is_file():
        manifest, blobs = _load_from_zip(FACTORIAL_INPUT_ZIP)
        source = FACTORIAL_INPUT_ZIP
    else:
        for candidate in sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True)):
            path = Path(candidate)
            try:
                probe = json.loads(path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if probe.get("schema") == "r6_factorial_data_v1":
                manifest, blobs = _load_from_directory(path)
                source = path.parent
                break

    if source is None:
        _diagnose_input_tree()
        raise AssertionError(
            "Batch 1 고정 데이터를 찾지 못했다. Batch 1이 만든 r6_factorial_data.zip을 "
            "Kaggle Dataset으로 올리고 이 노트북에 첨부하라. Kaggle이 zip을 풀어도 "
            "manifest.json과 여섯 JSONL이 있으면 인식한다 (위 진단 출력 참조)")

    assert manifest.get("schema") == "r6_factorial_data_v1", manifest
    assert manifest.get("members") and set(manifest["members"]) == set(EXPECTED_DATA_MEMBERS)
    for member in EXPECTED_DATA_MEMBERS:
        blob = blobs[member]
        digest = hashlib.sha256(blob).hexdigest()
        assert manifest["sha256"].get(member) == digest, f"hash 불일치: {member}"
        target = DATA_ROOT / member
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(blob)
    print(f"[restore] Batch 1 고정 데이터 복원: {source}")


## 7. STEP 3 — 학습 전 조작 점검과 Batch 1 artifact

전체 검사는 arm4가 arm1보다 넓고 arm5가 arm2보다 좁은지 같은 지표로 판정한다.
실패하면 다음 학습 셀은 실행하지 않는다. Batch 1은 통과한 여섯 파일만 manifest와 함께 zip으로 남긴다.

In [ ]:
# 조작이 실제 표적 분포에 나타나야만 통과한다.
run("python src/check_narrowness.py --require_generated")
print("[gate] STEP 3 arm4/arm5 조작 점검 통과")

for member in EXPECTED_DATA_MEMBERS:
    path = DATA_ROOT / member
    assert path.is_file(), f"학습 데이터가 없다: {path}"
    with path.open(encoding="utf-8") as fh:
        rows = [line for line in fh if line.strip()]
    assert len(rows) == 137, f"{path}: 기대 137행, 실제 {len(rows)}행"

if BATCH == 1:
    sha256 = {}
    for member in EXPECTED_DATA_MEMBERS:
        sha256[member] = hashlib.sha256((DATA_ROOT / member).read_bytes()).hexdigest()
    manifest = {
        "schema": "r6_factorial_data_v1",
        "seed": DATA_SEED,
        "target_pairs_per_arm": 411,
        "rows_per_round": 137,
        "members": EXPECTED_DATA_MEMBERS,
        "sha256": sha256,
    }
    with zipfile.ZipFile(FACTORIAL_OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for member in EXPECTED_DATA_MEMBERS:
            zf.write(DATA_ROOT / member, arcname=member)
        zf.writestr("manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))
    assert FACTORIAL_OUTPUT_ZIP.is_file() and FACTORIAL_OUTPUT_ZIP.stat().st_size > 0
    print(f"[artifact] {FACTORIAL_OUTPUT_ZIP} | {FACTORIAL_OUTPUT_ZIP.stat().st_size/1e6:.2f} MB")


## 8. 게이트 뒤 학습 · 평가

Batch 1은 여기서 멈추고 데이터 zip을 회수한다. Batch 2·3·4는 복원·재검증을 통과한 뒤 arm4와
arm5를 순서대로 학습하고, 각 arm 직후 결과 zip snapshot을 만든다. 세 배치의 차이는
`TRAIN_SEED` 하나뿐이며, 출력 파일명에 시드가 들어가므로 서로 덮어쓰지 않는다.

In [ ]:
import shutil

def snap(tag):
    """각 arm 직후 누적 결과를 보존해 뒤 세션 실패와 분리한다."""
    zp = shutil.make_archive(f"/kaggle/working/r6_results_{tag}", "zip",
                             "/kaggle/working/experiment/results")
    print(f"[snap] {zp} | {os.path.getsize(zp)/1e6:.2f} MB", flush=True)

if BATCH == 1:
    print("[stop] Batch 1은 학습하지 않는다. r6_factorial_data.zip을 먼저 회수하라.")
else:
    assert TRAIN_SEED is not None, "학습 배치인데 TRAIN_SEED가 없다"
    expected_outputs = [
        Path(f"results/eval_{arm}_r3_s{TRAIN_SEED}{OUT_SUFFIX}.json")
        for arm in ("arm4", "arm5")
    ]
    assert all(not path.exists() for path in expected_outputs), (
        f"기존 R6 출력을 덮어쓸 수 없다: {[str(p) for p in expected_outputs if p.exists()]}")
    for arm in ("arm4", "arm5"):
        print(f"\n===== {arm} x seed {TRAIN_SEED} =====", flush=True)
        run_session(arm, TRAIN_SEED, lenses=LENSES, eval_rounds=(3,),
                    probe=False, mmlu=False, max_new_tokens=MAX_NEW_TOKENS,
                    out_suffix=OUT_SUFFIX)
        snap(f"batch{BATCH}_{arm}_s{TRAIN_SEED}")


## 9. 결과 회수

`TAG`는 `BATCH`에서만 유도한다. Batch 1의 주 artifact는 데이터 zip이고, Batch 2·3·4는
두 arm의 최종 누적 results zip도 만든다. 배치 번호가 시드와 1:1이라 zip 이름도 겹치지 않는다.

In [ ]:
# TAG를 손으로 따로 적지 않아 배치 내용과 zip 이름이 어긋나지 않게 한다.
TAG = f"batch{BATCH}"
if BATCH == 1:
    assert FACTORIAL_OUTPUT_ZIP.is_file(), FACTORIAL_OUTPUT_ZIP
    print("data zip:", FACTORIAL_OUTPUT_ZIP,
          "| size MB:", round(FACTORIAL_OUTPUT_ZIP.stat().st_size / 1e6, 2))
else:
    zp = shutil.make_archive(f"/kaggle/working/r6_results_{TAG}", "zip",
                             "/kaggle/working/experiment/results")
    print("results zip:", zp, "| size MB:", round(os.path.getsize(zp) / 1e6, 2))


## 10. 최종 게이트

네 배치 모두 CPU 조작 점검과 137행×6 파일을 다시 확인한다. 학습 배치(2·3·4)는 접미사, schema,
환경 stamp, 렌즈 순서, 생성 상한, 표본 수와 per-record 생성 메타데이터까지 검사한다.
기대 파일명은 `TRAIN_SEED`에서 유도하므로 배치마다 다른 두 개다.

In [ ]:
import json, os

# 마지막 셀에서도 조작 점검을 다시 실행해 학습 artifact와 판정을 함께 남긴다.
run("python src/check_narrowness.py --require_generated")
for member in EXPECTED_DATA_MEMBERS:
    path = DATA_ROOT / member
    with path.open(encoding="utf-8") as fh:
        assert sum(1 for line in fh if line.strip()) == 137, path

if BATCH == 1:
    assert FACTORIAL_OUTPUT_ZIP.is_file() and FACTORIAL_OUTPUT_ZIP.stat().st_size > 0
    print("[gate] R6 BATCH 1 통과 — 조작 점검 + 고정 데이터 zip")
else:
    N = 313
    expected_lenses = ["none", "agreeable", "principled", "agreeable_para1"]
    expected = [
        f"results/eval_{arm}_r3_s{TRAIN_SEED}_r6_factorial_env76.json"
        for arm in ("arm4", "arm5")
    ]
    assert len(expected) == len(set(expected)), expected
    assert all(path.endswith(f"_r6_factorial_env76.json") for path in expected), expected
    assert '_r6_factorial_env76'.startswith("_") and '_r6_factorial_env76' != ""

    for path in expected:
        assert os.path.exists(path), f"결과가 없다: {path}"
        ev = json.load(open(path, encoding="utf-8"))
        assert ev.get("schema") == "r2", f"{path}: 스키마={ev.get('schema')}"
        assert ev.get("max_new_tokens") == 256, (
            f"{path}: max_new_tokens={ev.get('max_new_tokens')}")
        assert ev.get("out_suffix") == '_r6_factorial_env76', (
            f"{path}: out_suffix={ev.get('out_suffix')}")
        assert ev.get("env", {}).get("unsloth") == '2026.7.6', (
            f"{path}: unsloth={ev.get('env', {}).get('unsloth')}")
        assert ev.get("lenses") == expected_lenses, f"{path}: 렌즈={ev.get('lenses')}"
        assert list(ev.get("conditions", {})) == expected_lenses, (
            f"{path}: condition 순서/목록 불일치")

        for lens in expected_lenses:
            condition = ev["conditions"][lens]
            records = condition["records"]
            assert condition["n"] == N and len(records) == N, (
                f"{path}/{lens}: n={condition.get('n')} records={len(records)}")
            assert all({"hit_cap", "n_new_tokens", "finish_reason"} <= set(record)
                       for record in records), f"{path}/{lens}: 생성 메타데이터 누락"
            assert all(0 < record["n_new_tokens"] <= 256 for record in records), (
                f"{path}/{lens}: 생성 토큰 수 범위 오류")
            n_hit_cap = sum(bool(record["hit_cap"]) for record in records)
            assert condition.get("n_hit_cap") == n_hit_cap, (
                f"{path}/{lens}: cap 집계 불일치")
            assert all(record["finish_reason"] in
                       {"eos_token", "max_new_tokens", "stopping_criteria"}
                       for record in records), f"{path}/{lens}: 종료 이유 오류"
            print(f"  {os.path.basename(path)}/{lens} cap={n_hit_cap}/{N}")

    print(f"[gate] R6 BATCH {BATCH} 통과 — 학습 시드 {TRAIN_SEED}, "
          f"{len(expected)}개 eval 파일")
